# Homework 3 — Machine Learning for Classification

Machine Learning Zoomcamp 2026

## 1. Load and inspect the data

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [3]:
df = pd.read_csv('../00-data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
df.shape

(5000, 9)

In [5]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

The categorical missing values are replaced with `NA`. Numerical missing values are replaced with `0.0`, as requested.

In [6]:
categorical = df.select_dtypes(include=['object']).columns.tolist()
numerical = df.select_dtypes(include=['number']).columns.drop('converted').tolist()

print('Categorical:', categorical)
print('Numerical:', numerical)

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


C:\Users\khalid\AppData\Local\Temp\ipykernel_29776\2221264711.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include=['object']).columns.tolist()


In [7]:
df[categorical] = df[categorical].fillna('NA')
df[numerical] = df[numerical].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

## Question 1 — Mode of industry

Find the most frequent value of `industry` after filling the missing values.

In [17]:
df['industry'].value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

In [18]:
industry_mode = df['industry'].mode()[0]
print('Answer 1:', industry_mode)

Answer 1: technology


## Question 2 — Correlation

Compute the correlation matrix for the numerical input features. Compare only the four pairs provided in the question.

In [19]:
correlation_matrix = df[numerical].corr()
correlation_matrix.round(3)

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


In [20]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]

for first, second in pairs:
    print(f'{first} and {second}: {correlation_matrix.loc[first, second]:.4f}')

interaction_count and lead_score: 0.9157
number_of_courses_viewed and lead_score: 0.7572
number_of_courses_viewed and interaction_count: 0.7216
annual_income and interaction_count: 0.1228


In [21]:
best_pair = max(pairs, key=lambda pair: abs(correlation_matrix.loc[pair[0], pair[1]]))
print('Answer 2:', best_pair)

Answer 2: ('interaction_count', 'lead_score')


## 2. Split the dataset

Use the exact train/validation/test split required by Homework 3. The target is excluded from the input features.

In [22]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [23]:
y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

X_train = df_train.drop(columns=['converted'])
X_val = df_val.drop(columns=['converted'])
X_test = df_test.drop(columns=['converted'])

print('Training:', X_train.shape)
print('Validation:', X_val.shape)
print('Test:', X_test.shape)

Training: (3000, 8)
Validation: (1000, 8)
Test: (1000, 8)


## Question 3 — Mutual information

Calculate the mutual information between the target and each categorical feature using only the training data. Round to two decimal places for reporting.

In [24]:
mi_scores = {}

for column in categorical:
    score = mutual_info_score(y_train, X_train[column])
    mi_scores[column] = score
    print(column, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [25]:
best_mi_feature = max(mi_scores, key=mi_scores.get)
print('Answer 3:', best_mi_feature)

Answer 3: lead_source


## Question 4 — Logistic regression

Apply one-hot encoding with `DictVectorizer`. Fit the vectorizer on training data only, then apply it to the validation data.

In [26]:
train_dicts = X_train.to_dict(orient='records')
val_dicts = X_val.to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train_encoded = dv.fit_transform(train_dicts)
X_val_encoded = dv.transform(val_dicts)

print(X_train_encoded.shape, X_val_encoded.shape)

(3000, 28) (1000, 28)


In [27]:
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train_encoded, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [28]:
y_pred = model.predict(X_val_encoded)
original_accuracy = accuracy_score(y_val, y_pred)

print('Validation accuracy (unrounded):', original_accuracy)
print('Answer 4:', round(original_accuracy, 2))

Validation accuracy (unrounded): 0.645
Answer 4: 0.65


## Question 5 — Feature elimination

Remove each of the three specified features, retrain the model and compare unrounded accuracies with the original. The difference is *original accuracy minus accuracy without the feature*; it can be negative.

In [29]:
features_to_remove = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

In [30]:
feature_results = []

for feature in features_to_remove:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    dv_reduced = DictVectorizer(sparse=False)
    train_reduced = dv_reduced.fit_transform(X_train_reduced.to_dict(orient='records'))
    val_reduced = dv_reduced.transform(X_val_reduced.to_dict(orient='records'))

    model_reduced = LogisticRegression(
        solver='liblinear', C=1.0, max_iter=1000, random_state=42
    )
    model_reduced.fit(train_reduced, y_train)

    reduced_predictions = model_reduced.predict(val_reduced)
    reduced_accuracy = accuracy_score(y_val, reduced_predictions)
    difference = original_accuracy - reduced_accuracy

    feature_results.append({
        'removed_feature': feature,
        'accuracy': reduced_accuracy,
        'difference': difference
    })

In [31]:
feature_results_df = pd.DataFrame(feature_results)
feature_results_df

,removed_feature,accuracy,difference
0,lead_source,0.642,0.003
1,number_of_courses_viewed,0.643,0.002
2,interaction_count,0.601,0.044


In [32]:
answer_5 = feature_results_df.loc[feature_results_df['difference'].idxmin(), 'removed_feature']
print('Answer 5:', answer_5)

Answer 5: number_of_courses_viewed


## Question 6 — Regularization parameter C

Fit the full-feature logistic regression model at each specified `C`, evaluate validation accuracy and report it to three decimal places. Break ties by selecting the smallest `C`.

In [33]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]
c_results = []

In [34]:
for C in C_values:
    tuned_model = LogisticRegression(
        solver='liblinear', C=C, max_iter=1000, random_state=42
    )
    tuned_model.fit(X_train_encoded, y_train)

    tuned_predictions = tuned_model.predict(X_val_encoded)
    accuracy = accuracy_score(y_val, tuned_predictions)
    c_results.append({'C': C, 'accuracy': accuracy, 'rounded_accuracy': round(accuracy, 3)})

    print(f'C={C:g}: accuracy={accuracy:.3f}')

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645


In [35]:
c_results_df = pd.DataFrame(c_results)
c_results_df

,C,accuracy,rounded_accuracy
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


In [36]:
best_accuracy = c_results_df['rounded_accuracy'].max()
best_C = c_results_df.loc[
    c_results_df['rounded_accuracy'] == best_accuracy, 'C'
].min()

print('Answer 6:', best_C)

Answer 6: 0.001


## Final answers

These values come from the calculations above and can be submitted on the Homework 3 course page.

In [37]:
print('Q1:', industry_mode)
print('Q2:', ' and '.join(best_pair))
print('Q3:', best_mi_feature)
print('Q4:', round(original_accuracy, 2))
print('Q5:', answer_5)
print('Q6:', f'{best_C:g}')

Q1: technology
Q2: interaction_count and lead_score
Q3: lead_source
Q4: 0.65
Q5: number_of_courses_viewed
Q6: 0.001
